# 03 - Extração LLM (Principal)

Este é o **notebook principal** para extração CIMO usando LLMs.

## Modos de Operação
- **Interativo (Piloto):** Processa 1 artigo, mostra resultado, espera aprovação
- **Batch:** Processa todos automaticamente, só para se falhar validação

## Estratégia Híbrida
1. Conteúdo híbrido (texto + imagens seletivas) preparado no notebook 02
2. Claude (Anthropic) → Extração inicial
3. Validação local (12 regras)
4. GPT (OpenAI) → Repair se necessário

## 1. Setup e Configuração

In [1]:
import sys
import json
from pathlib import Path
from IPython.display import display, Markdown, HTML

NOTEBOOK_DIR = Path.cwd()
SYSTEM_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = SYSTEM_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from config import paths, llm_config, load_mapping, update_mapping_status, get_article_by_id
from pdf_vision import get_hybrid_content_for_anthropic, get_hybrid_content_for_openai, encode_image_base64
from llm_client import LLMClient, extract_yaml_from_response
from validators import validate_yaml, ValidationResult

print("✅ Módulos carregados")

✅ Módulos carregados


In [2]:
# DIAGNÓSTICO - Verificar chaves carregadas
import os
from dotenv import load_dotenv

# Recarregar .env forçadamente
env_path = SYSTEM_DIR / ".env"
load_dotenv(env_path, override=True)

# Mostrar chaves (parcialmente, por segurança)
anthropic_key = os.getenv("ANTHROPIC_API_KEY", "NÃO ENCONTRADA")
openai_key = os.getenv("OPENAI_API_KEY", "NÃO ENCONTRADA")

print(f"📁 Arquivo .env: {env_path}")
print(f"   Existe: {env_path.exists()}")
print()
print(f"🔑 ANTHROPIC_API_KEY:")
print(f"   Início: {anthropic_key[:25]}...")
print(f"   Final: ...{anthropic_key[-15:]}")
print()
print(f"🔑 OPENAI_API_KEY:")
print(f"   Início: {openai_key[:20]}...")
print(f"   Final: ...{openai_key[-10:]}")

📁 Arquivo .env: C:\Users\Leonardo\Documents\Computing\Projeto_Mestrado\files\files\articles\00 Dados RSL\system\.env
   Existe: True

🔑 ANTHROPIC_API_KEY:
   Início: sk-ant-api03-QhY8t6OXHf3f...
   Final: ...ec9MQw-zQ4BqQAA

🔑 OPENAI_API_KEY:
   Início: sk-proj-7rCE7vYn7A11...
   Final: ...ItfdJ2K_kA


## 2. Parâmetros de Execução

⚙️ **Configure aqui antes de executar**

In [3]:
# ========== CONFIGURAR AQUI ==========

MODO = "batch"              # ou "interativo"
ARTIGO_TESTE = "ART_005"    # pode deixar qualquer um; no batch não importa

PROVIDER_EXTRACAO = "anthropic"
PROVIDER_REPAIR = "ollama"
MAX_REPAIR_ATTEMPTS = 3

# Fallback automático (desligado porque já estamos usando Anthropic)
ENABLE_FALLBACK = False
FALLBACK_PROVIDER = "anthropic"  # pode deixar, mas não será usado

# =====================================

print(f"📋 Configuração:")
print(f"   Modo: {MODO}")
print(f"   Artigo teste: {ARTIGO_TESTE}")
print(f"   Extração (primária): {PROVIDER_EXTRACAO}")
print(f"   Repair: {PROVIDER_REPAIR}")
print(f"   Fallback: {FALLBACK_PROVIDER if ENABLE_FALLBACK else 'OFF'}")

📋 Configuração:
   Modo: batch
   Artigo teste: ART_005
   Extração (primária): anthropic
   Repair: ollama
   Fallback: OFF


## 3. Carregar Recursos

In [4]:
# Carregar prompt do Guia
with open(paths.GUIA_PROMPT, "r", encoding="utf-8") as f:
    GUIA_PROMPT = f.read()

print(f"✅ Prompt carregado: {len(GUIA_PROMPT)} caracteres (~{len(GUIA_PROMPT)//4} tokens)")

# Inicializar cliente LLM
client = LLMClient()
print(f"✅ Cliente LLM inicializado")
print(f"   Anthropic: {'✅' if client.anthropic else '❌'}")
print(f"   OpenAI: {'✅' if client.openai else '❌'}")

✅ Prompt carregado: 9382 caracteres (~2345 tokens)
✅ Cliente LLM inicializado
   Anthropic: ✅
   OpenAI: ✅


## 4. Funções de Extração

In [5]:
def load_hybrid_content(artigo_id: str, provider: str = "anthropic") -> list[dict]:
    """
    Carrega conteúdo híbrido de um artigo no formato do provider.
    """
    work_dir = paths.WORK / artigo_id
    hybrid_file = work_dir / "hybrid.json"
    texts_file = work_dir / "texts.json"
    pages_dir = work_dir / "pages"
    
    if not hybrid_file.exists():
        raise ValueError(f"Arquivo hybrid.json não encontrado. Execute notebook 02 primeiro.")
    
    # Carregar metadados
    with open(hybrid_file, "r", encoding="utf-8") as f:
        hybrid_meta = json.load(f)
    
    # Carregar textos
    with open(texts_file, "r", encoding="utf-8") as f:
        texts = json.load(f)
    
    # Reconstruir estrutura de páginas
    pages = []
    for page_info in hybrid_meta["pages_info"]:
        page_num = page_info["page_num"]
        
        if page_info["type"] == "text":
            pages.append({
                "page_num": page_num,
                "type": "text",
                "content": texts.get(str(page_num), "")
            })
        elif page_info["type"] == "image":
            pages.append({
                "page_num": page_num,
                "type": "image",
                "path": Path(page_info["path"])
            })
    
    stats = hybrid_meta["stats"]
    print(f"   📊 Conteúdo: {stats['text_pages']} texto, {stats['image_pages']} imagem, {stats['skipped_pages']} ignoradas")
    
    # Converter para formato do provider
    hybrid_result = {"pages": pages}
    
    if provider == "anthropic":
        return get_hybrid_content_for_anthropic(hybrid_result)
    else:
        return get_hybrid_content_for_openai(hybrid_result)


def extract_article(artigo_id: str) -> tuple[str, ValidationResult]:
    """
    Executa extração completa de um artigo usando conteúdo híbrido.
    
    Returns:
        (yaml_content, validation_result)
    """
    work_dir = paths.get_article_work_dir(artigo_id)
    
    # Carregar conteúdo híbrido
    print(f"   🔄 Carregando conteúdo híbrido...")
    content = load_hybrid_content(artigo_id, PROVIDER_EXTRACAO)
    
    # Chamar LLM para extração
    print(f"   🤖 Chamando {PROVIDER_EXTRACAO}...")
    response = client.extract_with_hybrid(
        content=content,
        prompt=GUIA_PROMPT,
        artigo_id=artigo_id,
        provider=PROVIDER_EXTRACAO
    )
    
    # Extrair YAML da resposta
    yaml_content = extract_yaml_from_response(response)
    
    # Salvar draft
    draft_path = work_dir / "draft.yaml"
    with open(draft_path, "w", encoding="utf-8") as f:
        f.write(yaml_content)
    print(f"   💾 Draft salvo")
    
    # Validar
    print(f"   🔍 Validando...")
    result = validate_yaml(yaml_content)
    
    return yaml_content, result


def repair_yaml_loop(
    yaml_content: str,
    errors: list[str],
    artigo_id: str,
    max_attempts: int = 3
) -> tuple[str, ValidationResult]:
    """
    Loop de repair até validar ou esgotar tentativas.
    """
    work_dir = paths.get_article_work_dir(artigo_id)
    
    for attempt in range(1, max_attempts + 1):
        print(f"   🔧 Repair tentativa {attempt}/{max_attempts}...")
        
        repaired = client.repair_yaml(
            yaml_content=yaml_content,
            errors=errors,
            provider=PROVIDER_REPAIR
        )
        
        repaired_yaml = extract_yaml_from_response(repaired)
        
        # Salvar tentativa
        repair_path = work_dir / f"repair_attempt_{attempt:02d}.yaml"
        with open(repair_path, "w", encoding="utf-8") as f:
            f.write(repaired_yaml)
        
        # Validar
        result = validate_yaml(repaired_yaml)
        
        if result.is_valid:
            print(f"   ✅ Repair bem-sucedido na tentativa {attempt}")
            return repaired_yaml, result
        
        yaml_content = repaired_yaml
        errors = result.errors
        print(f"   ⚠️ Ainda {len(errors)} erros")
    
    print(f"   ❌ Repair falhou após {max_attempts} tentativas")
    return yaml_content, result

## 5. Funções de Exibição

In [6]:
def display_yaml(yaml_content: str, max_lines: int = 100):
    """Exibe YAML com syntax highlighting."""
    lines = yaml_content.split("\n")
    if len(lines) > max_lines:
        truncated = "\n".join(lines[:max_lines])
        truncated += f"\n\n... ({len(lines) - max_lines} linhas omitidas)"
        display(Markdown(f"```yaml\n{truncated}\n```"))
    else:
        display(Markdown(f"```yaml\n{yaml_content}\n```"))


def display_validation(result: ValidationResult):
    """Exibe resultado da validação."""
    if result.is_valid:
        display(HTML("<h3 style='color: green'>✅ VALIDAÇÃO: APROVADO</h3>"))
    else:
        display(HTML("<h3 style='color: red'>❌ VALIDAÇÃO: REPROVADO</h3>"))
        html = "<b>Erros:</b><ul>"
        for e in result.errors:
            html += f"<li style='color: red'>{e}</li>"
        html += "</ul>"
        display(HTML(html))
    
    if result.warnings:
        html = "<b>Avisos:</b><ul>"
        for w in result.warnings:
            html += f"<li style='color: orange'>{w}</li>"
        html += "</ul>"
        display(HTML(html))
    
    if result.rules_passed:
        display(HTML(f"<p>Regras OK: {result.rules_passed}</p>"))
    if result.rules_failed:
        display(HTML(f"<p style='color: red'>Regras violadas: {result.rules_failed}</p>"))

## 6. Funções de Persistência

In [7]:
def save_approved(artigo_id: str, yaml_content: str):
    """Salva YAML aprovado e atualiza mapping."""
    # Salvar em work/
    work_dir = paths.get_article_work_dir(artigo_id)
    work_path = work_dir / "extraction.yaml"
    with open(work_path, "w", encoding="utf-8") as f:
        f.write(yaml_content)
    
    # Copiar para yamls/
    final_path = paths.YAMLS / f"{artigo_id}.yaml"
    with open(final_path, "w", encoding="utf-8") as f:
        f.write(yaml_content)
    
    # Salvar validação
    result = validate_yaml(yaml_content)
    import json
    validation_path = work_dir / "validation.json"
    with open(validation_path, "w", encoding="utf-8") as f:
        json.dump(result.to_dict(), f, indent=2)
    
    # Atualizar mapping
    update_mapping_status(paths.MAPPING_CSV, artigo_id, True, "Aprovado")
    
    print(f"✅ {artigo_id} salvo e aprovado!")
    print(f"   YAML: {final_path}")

---
## 7. EXTRAÇÃO INTERATIVA (Piloto)

Execute as células abaixo para processar o artigo teste.

In [ ]:
# Verificar artigo
article = get_article_by_id(paths.MAPPING_CSV, ARTIGO_TESTE)

if article:
    print(f"🎯 Artigo selecionado:")
    print(f"   ID: {article['ArtigoID']}")
    print(f"   Arquivo: {article['Arquivo'][:60]}...")
    print(f"   Status: {article['Processado']} - {article['Status']}")
else:
    print(f"❌ Artigo não encontrado: {ARTIGO_TESTE}")

In [ ]:
# EXECUTAR EXTRAÇÃO (com fallback automático)
print(f"🚀 Extraindo {ARTIGO_TESTE}...")
print("="*50)

# Carregar conteúdo híbrido nos dois formatos
# - openai: serve tanto para provider=openai quanto provider=ollama (OpenAI-compatible)
# - anthropic: usado só no fallback (Claude)
content_openai = load_hybrid_content(ARTIGO_TESTE, provider="openai")
content_anthropic = load_hybrid_content(ARTIGO_TESTE, provider="anthropic")

if PROVIDER_EXTRACAO == "ollama" and ENABLE_FALLBACK:
    yaml_content, result = client.extract_validated_with_fallback(
        content_openai=content_openai,
        content_anthropic=content_anthropic,
        prompt=GUIA_PROMPT,
        artigo_id=ARTIGO_TESTE,
        primary_provider="ollama",
        fallback_provider=FALLBACK_PROVIDER,
        repair_provider=PROVIDER_REPAIR,
        max_repair_attempts=MAX_REPAIR_ATTEMPTS,
        enable_fallback=True,
    )
elif PROVIDER_EXTRACAO == "openai" and ENABLE_FALLBACK:
    yaml_content, result = client.extract_validated_with_fallback(
        content_openai=content_openai,
        content_anthropic=content_anthropic,
        prompt=GUIA_PROMPT,
        artigo_id=ARTIGO_TESTE,
        primary_provider="openai",
        fallback_provider=FALLBACK_PROVIDER,
        repair_provider=PROVIDER_REPAIR,
        max_repair_attempts=MAX_REPAIR_ATTEMPTS,
        enable_fallback=True,
    )
else:
    # Modo antigo (sem fallback automático)
    yaml_content, result = extract_article(ARTIGO_TESTE)

print("\n" + "="*50)
print("RESULTADO DA EXTRAÇÃO")
print("="*50)


In [ ]:
# Exibir YAML extraído
display_yaml(yaml_content)

In [ ]:
# Exibir validação
display_validation(result)

### 7.1 Ações

In [ ]:
# APROVAR - Execute se o resultado estiver OK
save_approved(ARTIGO_TESTE, yaml_content)

In [ ]:
# REPARAR - Execute se houver erros de validação
# yaml_content, result = repair_yaml_loop(yaml_content, result.errors, ARTIGO_TESTE)
# display_yaml(yaml_content)
# display_validation(result)

In [ ]:
# PULAR - Execute para marcar como pulado e seguir adiante
# update_mapping_status(paths.MAPPING_CSV, ARTIGO_TESTE, False, "Pulado")
# print("⏭️ Artigo pulado")

---
## 8. MODO BATCH

⚠️ **Execute apenas após validar o prompt no modo piloto.**

In [8]:
# Verificar artigos pendentes
from config import get_pending_articles

pending = get_pending_articles(paths.MAPPING_CSV)
print(f"📊 Artigos pendentes: {len(pending)}")

for p in pending[:5]:
    print(f"   {p['ArtigoID']}: {p['Arquivo'][:40]}...")
if len(pending) > 5:
    print(f"   ... e mais {len(pending) - 5}")

📊 Artigos pendentes: 13
   ART_011: Comprehensive Decision Framework Combini...
   ART_013: Data-Driven Optimization for Oilfield Op...
   ART_015: Development of a decision support system...
   ART_016: Digital Twin Applications in EPC Project...
   ART_017: Efficiency enhancement of energy supply ...
   ... e mais 8


In [ ]:
# Switch de segurança (evita execuções acidentais)
RUN_BATCH = True  # mude para True quando estiver pronto

# Opcional: mini-batch controlado.
# - Deixe None para processar TODOS os pendentes.
# - Ou defina um set/list com ids, ex: {"ART_003","ART_004","ART_005","ART_006","ART_007"}
TARGET_IDS = None

if TARGET_IDS:
    target_set = set(TARGET_IDS)
    pending = [a for a in pending if a.get("ArtigoID") in target_set]

if not RUN_BATCH:
    print("🛑 Batch não executado (RUN_BATCH=False).")
    print("   Quando estiver pronto, defina RUN_BATCH=True e reexecute esta célula.")
    if TARGET_IDS:
        print(f"   (Mini-batch) TARGET_IDS={sorted(set(TARGET_IDS))}")
    else:
        print("   TARGET_IDS=None (rodaria todos os pendentes)")
else:
    from tqdm.notebook import tqdm

    print(f"🚀 Processando {len(pending)} artigos em batch...")
    if TARGET_IDS:
        print(f"   (Mini-batch) TARGET_IDS={sorted(set(TARGET_IDS))}")
    print("="*60)

    batch_results = []

    for article in tqdm(pending, desc="Extraindo"):
        artigo_id = article["ArtigoID"]
        print(f"\n▶️ {artigo_id}")

        try:
            # Preparar conteúdo híbrido (necessário para providers com fallback)
            content_openai = load_hybrid_content(artigo_id, provider="openai")
            content_anthropic = load_hybrid_content(artigo_id, provider="anthropic")

            if PROVIDER_EXTRACAO in ["ollama", "openai"] and ENABLE_FALLBACK:
                yaml_content, result = client.extract_validated_with_fallback(
                    content_openai=content_openai,
                    content_anthropic=content_anthropic,
                    prompt=GUIA_PROMPT,
                    artigo_id=artigo_id,
                    primary_provider=PROVIDER_EXTRACAO,
                    fallback_provider=FALLBACK_PROVIDER,
                    repair_provider=PROVIDER_REPAIR,
                    max_repair_attempts=MAX_REPAIR_ATTEMPTS,
                    enable_fallback=True,
                    # Quote-QC automático (só reextrai quotes se detectar mismatch em páginas-texto)
                    enable_quotes_qc=True,
                )
            else:
                # Modo antigo (sem fallback automático)
                yaml_content, result = extract_article(artigo_id)

                # Reparar se necessário
                if not result.is_valid:
                    yaml_content, result = repair_yaml_loop(
                        yaml_content, result.errors, artigo_id, MAX_REPAIR_ATTEMPTS
                    )

            # Salvar resultado
            if result.is_valid:
                save_approved(artigo_id, yaml_content)
                batch_results.append({"id": artigo_id, "status": "approved"})
            else:
                update_mapping_status(paths.MAPPING_CSV, artigo_id, False, "Falhou validação")
                batch_results.append({"id": artigo_id, "status": "failed", "errors": result.errors})
                print(f"   ❌ Falhou: {result.errors[:2]}")

        except Exception as e:
            update_mapping_status(paths.MAPPING_CSV, artigo_id, False, f"Erro: {str(e)[:50]}")
            batch_results.append({"id": artigo_id, "status": "error", "error": str(e)})
            print(f"   ❌ Erro: {e}")

    # Resumo
    approved = len([r for r in batch_results if r["status"] == "approved"])
    failed = len([r for r in batch_results if r["status"] == "failed"])
    errors = len([r for r in batch_results if r["status"] == "error"])

    print("\n" + "="*60)
    print("RESUMO DO BATCH")
    print("="*60)
    print(f"✅ Aprovados: {approved}")
    print(f"❌ Falharam validação: {failed}")
    print(f"💥 Erros: {errors}")


---
## 9. Status Final

In [9]:
# Verificar status geral
import pandas as pd

df = pd.read_csv(paths.MAPPING_CSV)

print("📊 Status Geral:")
print(f"   Total: {len(df)}")
print(f"   Processados: {len(df[df['Processado'] == 'Sim'])}")
print(f"   Pendentes: {len(df[df['Processado'] != 'Sim'])}")

# Status breakdown
print("\n📋 Detalhamento:")
print(df['Status'].value_counts())

# YAMLs gerados
yamls = list(paths.YAMLS.glob("*.yaml"))
print(f"\n📁 YAMLs aprovados: {len(yamls)}")

if len(yamls) > 0:
    print("\n✅ Pronto para 05_Consolidacao.ipynb")

📊 Status Geral:
   Total: 38
   Processados: 25
   Pendentes: 13

📋 Detalhamento:
Status
Aprovado            25
Falhou validação    13
Name: count, dtype: int64

📁 YAMLs aprovados: 38

✅ Pronto para 05_Consolidacao.ipynb
